# Lab 06: Inverse Dynamics & Static Optimization**Computational Sensorimotor Control — Week 6****Task:** Trace a semicircular arc (R = 6 cm) starting from Q_REF = (55°, 75°), with one via-point at the tip of the arc, in 800 ms. Two conditions: no external load and a constant 3 N force in −y.**Objectives:**- Run EPH, VITE, and min-jerk on the arc task and measure path deviation- Find optimal EPH parameters (R-shift and C) via grid search — and understand why the brain probably doesn't do this- Build the inverse dynamics pipeline step by step- Implement continuous equilibrium tracking and understand its limitations- Solve the muscle redundancy problem with static optimization (Crowninshield & Brand 1981)- Compare ID and EPH muscle activation patterns head to head

In [ ]:
!pip3 install git+https://github.com/tarkeshsingh/computational-sensorimotor-control.git#subdirectory=smc -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize as scipy_minimize
plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 11,
                      'axes.grid': True, 'grid.alpha': 0.3})

from smc import (
    Arm, make_muscles, simulate_lambda,
    Q_REF, MUSCLE_DEFS,
    mass_matrix, coriolis, joint_accelerations, rk4_step,
    C_EXP, MU_LAMBDA,
)

arm = Arm()
ik  = arm.inverse_kinematics
fk  = arm.forward_kinematics
start_hand = fk(Q_REF)

# ── Constants ──
RADII    = [0.03, 0.06, 0.09]
R_ARC    = 0.06
T_MOVE   = 0.800
F_LOAD   = np.array([0.0, -3.0])
J_ref    = arm.jacobian(Q_REF)
tau_ext  = J_ref.T @ F_LOAD
perturb_3N = lambda t: tau_ext

# ── Geometry helpers ──
def arc_geometry(R):
    center = start_hand + np.array([R, 0])
    tip    = start_hand + np.array([R, R])
    end    = start_hand + np.array([2*R, 0])
    return center, tip, end

def dense_arc(R, n=500):
    center = start_hand + np.array([R, 0])
    th = np.linspace(np.pi, 0, n)
    return np.column_stack([center[0]+R*np.cos(th), center[1]+R*np.sin(th)])

def max_path_deviation(hand, desired):
    return max(np.linalg.norm(desired - hand[i], axis=1).min() for i in range(len(hand)))

# ── Lambda helper (same as Week 4) ──
def lambda_for_posture(q_target, C=0.25):
    muscles = make_muscles()
    lam = np.zeros(6)
    for j, m in enumerate(muscles):
        l_eq = m.length(q_target)
        shift = (abs(m.r_sh) + abs(m.r_el)) * C
        lam[j] = l_eq - shift
    return lam

# ── Min-jerk along the arc ──
def minjerk_arc(R, T=T_MOVE, dt=0.001):
    center = start_hand + np.array([R, 0])
    t = np.arange(0, T + dt, dt)
    tau = t / T
    s = 10*tau**3 - 15*tau**4 + 6*tau**5
    theta = np.pi * (1 - s)
    pos = np.column_stack([center[0]+R*np.cos(theta), center[1]+R*np.sin(theta)])
    return t, pos

# ── VITE with via-point ──
def vite_arc(R, G_amp=10, T=T_MOVE, dt=0.0002):
    _, tip, end = arc_geometry(R)
    alpha = 4.0 * G_amp
    thresh = 0.05 * np.linalg.norm(tip - start_hand)
    state = np.array([start_hand[0], start_hand[1], 0.0, 0.0])
    n = int(T / dt); t_arr = np.arange(n)*dt
    pos = np.zeros((n,2)); switched = False
    for i in range(n):
        pos[i] = state[:2]
        target = end if switched else tip
        if not switched and np.linalg.norm(pos[i]-tip) < thresh:
            switched = True; target = end
        def deriv(s, _t=target):
            return np.array([s[2], s[3],
                             alpha*(-s[2]+G_amp*(_t[0]-s[0])),
                             alpha*(-s[3]+G_amp*(_t[1]-s[1]))])
        state = rk4_step(state, deriv, dt)
    return t_arr, pos

# ── EPH arc with R-shift ──
def eph_arc(R, dy=0, C=0.25, perturb_fn=None, dt=0.0005):
    _, tip, end = arc_geometry(R)
    tip_s = tip + np.array([0, dy]); end_s = end + np.array([0, dy])
    try:
        q_tip = ik(tip_s[0], tip_s[1]); q_end = ik(end_s[0], end_s[1])
        if not arm.in_joint_limits(q_tip) or not arm.in_joint_limits(q_end): return None, None, None
    except: return None, None, None
    li = lambda_for_posture(Q_REF, C)
    l_tip = lambda_for_posture(q_tip, C); l_end = lambda_for_posture(q_end, C)
    ramp = T_MOVE / 2
    def lam_fn(t):
        if t < ramp: return li + (t/ramp)*(l_tip - li)
        elif t < 2*ramp: return l_tip + ((t-ramp)/ramp)*(l_end - l_tip)
        else: return l_end.copy()
    t_arr, states, hand, acts = simulate_lambda(lam_fn, T=1.2, dt=dt, perturb_fn=perturb_fn)
    return t_arr, hand, acts

print(f'Start: ({start_hand[0]*100:.1f}, {start_hand[1]*100:.1f}) cm')
_, tip6, end6 = arc_geometry(R_ARC)
print(f'6 cm arc — Tip: ({tip6[0]*100:.1f}, {tip6[1]*100:.1f}) cm, End: ({end6[0]*100:.1f}, {end6[1]*100:.1f}) cm')

---## Part 1: EPH on the Arc — Two Goals, Two Controllers (Lecture §1 — Figure 1)Run min-jerk, VITE, and EPH on the 6 cm arc. For EPH under 3 N load, use a **grid search** to find the optimal R-shift (dy) and C-command that minimize (a) endpoint error and (b) path deviation separately.**A note on the grid search:** The grid search explores hundreds of (dy, C) combinations to find the best pair for each goal. This is a brute-force computational tool — the brain almost certainly does not solve a new optimization for every novel task. Through repeated practice, the nervous system may learn appropriate R and C adjustments for familiar loads and trajectories, effectively building a lookup table from experience. But it has no principled way to generalize to untested conditions. This is a fundamental limitation of EPH that inverse dynamics (§3) does not share.**Your tasks:** Fill in the TODO lines to:1. Compute λ for the via-point (tip) posture2. Run EPH without load3. Run the grid search for endpoint-optimized EPH under 3 N load

In [ ]:
# ── Kinematic controllers (no muscles, no load dependence) ──
t_mj, pos_mj = minjerk_arc(R_ARC)
t_vt, pos_vt = vite_arc(R_ARC)
des6 = dense_arc(R_ARC)
_, tip6, end6 = arc_geometry(R_ARC)

pd_mj = max_path_deviation(pos_mj, des6) * 100
pd_vt = max_path_deviation(pos_vt, des6) * 100
ep_mj = np.linalg.norm(pos_mj[-1] - end6) * 100
ep_vt = np.linalg.norm(pos_vt[-1] - end6) * 100

# ── EPH: compute lambda values ──
C_CMD = 0.25
lam_start = lambda_for_posture(Q_REF, C_CMD)
q_tip = ik(tip6[0], tip6[1])

# TODO 1: Compute λ for the via-point (tip) posture with C_CMD
# ← YOUR CODE HERE

# ── EPH naive: no load ──
# TODO 2: Run eph_arc with R=R_ARC, dy=0, C=C_CMD, no perturbation
# ← YOUR CODE HERE

# ── EPH naive: 3N load (same parameters, no compensation) ──
t_eph3, h_eph3, a_eph3 = eph_arc(R_ARC, dy=0, C=C_CMD, perturb_fn=perturb_3N)

pd_eph0 = max_path_deviation(h_eph0, des6)*100
pd_eph3 = max_path_deviation(h_eph3, des6)*100
ep_eph0 = np.linalg.norm(h_eph0[-1] - end6)*100
ep_eph3 = np.linalg.norm(h_eph3[-1] - end6)*100

print(f'Min-jerk:      ep={ep_mj:.3f} cm, pd={pd_mj:.3f} cm')
print(f'VITE:          ep={ep_vt:.3f} cm, pd={pd_vt:.3f} cm')
print(f'EPH naive 0N:  ep={ep_eph0:.2f} cm, pd={pd_eph0:.2f} cm')
print(f'EPH naive 3N:  ep={ep_eph3:.2f} cm, pd={pd_eph3:.2f} cm')

In [ ]:
# ── Grid search for optimal EPH under 3 N load ──
# This explores ~150 (dy, C) combinations for each metric.
# The brain does NOT do this — it learns from experience.
# We use it here to find EPH's theoretical best performance.

def grid_search_eph(R, metric='endpoint', perturb_fn=perturb_3N):
    des = dense_arc(R)
    _, _, end_pt = arc_geometry(R)
    best_val = 999; best_dy = 0; best_C = 0.25
    for dy in np.linspace(0.0, 0.08, 12):
        for C in np.linspace(0.20, 0.60, 12):
            _, h, _ = eph_arc(R, dy=dy, C=C, perturb_fn=perturb_fn)
            if h is None: continue
            if metric == 'endpoint':
                val = np.linalg.norm(h[-1] - end_pt)
            else:
                val = max_path_deviation(h, des)
            if val < best_val:
                best_val = val; best_dy = dy; best_C = C
    return best_dy, best_C, best_val

# TODO 3: Run grid search optimizing for endpoint error under 3 N load
# ← YOUR CODE HERE

# Grid search optimizing for path deviation under 3 N load
dy_pd, C_pd, _ = grid_search_eph(R_ARC, metric='path')

# Run the optimized controllers
_, h_opt_ep, _ = eph_arc(R_ARC, dy=dy_ep, C=C_ep, perturb_fn=perturb_3N)
_, h_opt_pd, _ = eph_arc(R_ARC, dy=dy_pd, C=C_pd, perturb_fn=perturb_3N)

ep_opt_ep = np.linalg.norm(h_opt_ep[-1] - end6)*100
pd_opt_ep = max_path_deviation(h_opt_ep, des6)*100
ep_opt_pd = np.linalg.norm(h_opt_pd[-1] - end6)*100
pd_opt_pd = max_path_deviation(h_opt_pd, des6)*100

print(f'\nEPH opt endpoint 3N: dy={dy_ep*100:.1f}cm, C={C_ep:.2f} -> ep={ep_opt_ep:.2f}, pd={pd_opt_ep:.2f}')
print(f'EPH opt path 3N:     dy={dy_pd*100:.1f}cm, C={C_pd:.2f} -> ep={ep_opt_pd:.2f}, pd={pd_opt_pd:.2f}')
print(f'\nKey: different goals → different R & C parameters')

In [ ]:
# ═══════════════════════════════════════════════
# FIGURE 1 (Lecture §1): 3-panel comparison on 6 cm arc
# ═══════════════════════════════════════════════
NAVY='#1B2A4A'; TEAL='#2E86AB'; GREEN='#27AE60'; RED='#E74C3C'; ORANGE='#E67E22'; PURPLE='#8E44AD'

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel A: Hand paths
ax = axes[0]
ax.plot(des6[:,0]*100, des6[:,1]*100, 'gray', lw=2.5, alpha=0.35, label='Desired arc')
ax.plot(pos_mj[:,0]*100, pos_mj[:,1]*100, TEAL, lw=2.5, label='Min-jerk')
ax.plot(pos_vt[:,0]*100, pos_vt[:,1]*100, PURPLE, lw=2, ls='--', label='VITE')
ax.plot(h_eph0[:,0]*100, h_eph0[:,1]*100, GREEN, lw=2, label='EPH naive, 0 N')
ax.plot(h_eph3[:,0]*100, h_eph3[:,1]*100, RED, lw=2, label='EPH naive, 3 N')
ax.plot(h_opt_ep[:,0]*100, h_opt_ep[:,1]*100, ORANGE, lw=2, ls='-.', label='EPH opt. endpoint, 3 N')
ax.plot(h_opt_pd[:,0]*100, h_opt_pd[:,1]*100, NAVY, lw=2, ls=':', label='EPH opt. path, 3 N')
ax.plot(start_hand[0]*100, start_hand[1]*100, 'ko', ms=8, zorder=5)
ax.plot(tip6[0]*100, tip6[1]*100, 'k^', ms=7, zorder=5)
ax.plot(end6[0]*100, end6[1]*100, 'k*', ms=11, zorder=5)
ax.set_xlabel('X (cm)'); ax.set_ylabel('Y (cm)')
ax.set_title('A. Hand paths (R = 6 cm)'); ax.legend(fontsize=6.5); ax.set_aspect('equal')

# Panel B: Endpoint errors
ax = axes[1]
labels = ['MJ', 'VITE', 'EPH\n0N', 'EPH\n3N', 'EPH opt\nendpt', 'EPH opt\npath']
ep_vals = [ep_mj, ep_vt, ep_eph0, ep_eph3, ep_opt_ep, ep_opt_pd]
colors = [TEAL, PURPLE, GREEN, RED, ORANGE, NAVY]
bars = ax.bar(range(len(labels)), ep_vals, color=colors, alpha=0.8)
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel('Endpoint error (cm)'); ax.set_title('B. Endpoint error')
for b, v in zip(bars, ep_vals): ax.text(b.get_x()+b.get_width()/2, v+0.05, f'{v:.2f}', ha='center', fontsize=7)

# Panel C: Path deviations
ax = axes[2]
pd_vals = [pd_mj, pd_vt, pd_eph0, pd_eph3, pd_opt_ep, pd_opt_pd]
bars = ax.bar(range(len(labels)), pd_vals, color=colors, alpha=0.8)
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel('Max path deviation (cm)'); ax.set_title('C. Path deviation')
for b, v in zip(bars, pd_vals): ax.text(b.get_x()+b.get_width()/2, v+0.05, f'{v:.2f}', ha='center', fontsize=7)

plt.tight_layout(); plt.show()

---## Part 2: Path Deviation Across Arc Radii (Lecture §2 — Figure 2)Extend the comparison to all three arc radii (3, 6, 9 cm) under 3 N load, optimizing EPH for path deviation at each radius. This demonstrates that EPH's optimal parameters are trajectory-dependent — the brain's "lookup table" would need a different entry for each movement shape.**Your task:** Fill in the TODO line to run the grid search for each radius.

In [ ]:
# ── Run all controllers for 3 radii ──
results = {}
for R in RADII:
    des = dense_arc(R); _, tip_r, end_r = arc_geometry(R)
    # Kinematic controllers
    _, pm = minjerk_arc(R); _, pv = vite_arc(R)
    # EPH naive
    _, h0, _ = eph_arc(R, dy=0, C=0.25)
    _, h3, _ = eph_arc(R, dy=0, C=0.25, perturb_fn=perturb_3N)
    # TODO 1: Run grid search for path-deviation-optimized EPH under 3 N load
    # ← YOUR CODE HERE
    _, h_opt, _ = eph_arc(R, dy=dy_opt, C=C_opt, perturb_fn=perturb_3N)
    results[R] = dict(des=des, tip=tip_r, end=end_r,
        pm=pm, pv=pv, h0=h0, h3=h3, h_opt=h_opt,
        pd_mj=max_path_deviation(pm,des)*100, pd_vt=max_path_deviation(pv,des)*100,
        pd_0N=max_path_deviation(h0,des)*100, pd_3N=max_path_deviation(h3,des)*100,
        pd_opt=max_path_deviation(h_opt,des)*100, dy=dy_opt, C=C_opt)

print(f'{"R":>4} {"MJ":>8} {"VITE":>8} {"EPH 0N":>8} {"EPH 3N":>8} {"EPH opt":>8} {"dy(cm)":>7} {"C":>5}')
for R in RADII:
    r = results[R]
    print(f'{R*100:3.0f}  {r["pd_mj"]:7.2f} {r["pd_vt"]:7.2f} {r["pd_0N"]:7.2f} {r["pd_3N"]:7.2f} {r["pd_opt"]:7.2f}  {r["dy"]*100:5.1f} {r["C"]:4.2f}')

In [ ]:
# ═══════════════════════════════════════════════
# FIGURE 2 (Lecture §2): Three arcs, path deviation
# ═══════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, R in enumerate(RADII):
    r = results[R]; ax = axes[idx]
    ax.plot(r['des'][:,0]*100, r['des'][:,1]*100, 'gray', lw=2.5, alpha=0.35, label='Desired')
    ax.plot(r['pm'][:,0]*100, r['pm'][:,1]*100, TEAL, lw=2, label=f"MJ ({r['pd_mj']:.2f})")
    ax.plot(r['pv'][:,0]*100, r['pv'][:,1]*100, PURPLE, lw=2, ls='--', label=f"VITE ({r['pd_vt']:.2f})")
    ax.plot(r['h0'][:,0]*100, r['h0'][:,1]*100, GREEN, lw=1.5, alpha=0.6, label=f"EPH 0N ({r['pd_0N']:.2f})")
    ax.plot(r['h3'][:,0]*100, r['h3'][:,1]*100, RED, lw=1.5, alpha=0.6, label=f"EPH 3N ({r['pd_3N']:.2f})")
    ax.plot(r['h_opt'][:,0]*100, r['h_opt'][:,1]*100, ORANGE, lw=2.5, ls='-.',
            label=f"EPH opt ({r['pd_opt']:.2f})")
    ax.plot(start_hand[0]*100, start_hand[1]*100, 'ko', ms=8, zorder=5)
    ax.plot(r['tip'][0]*100, r['tip'][1]*100, 'k^', ms=7, zorder=5)
    ax.plot(r['end'][0]*100, r['end'][1]*100, 'k*', ms=11, zorder=5)
    ax.set_xlabel('X (cm)')
    if idx == 0: ax.set_ylabel('Y (cm)')
    ax.set_title(f'R = {R*100:.0f} cm', fontweight='bold')
    ax.legend(fontsize=6); ax.set_aspect('equal')

plt.suptitle('Path deviation across arc radii (EPH optimized for path under 3 N load)', fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

---## Part 3: The Inverse Dynamics Pipeline (Lecture §3 — Figure 3)Build the ID pipeline step by step on the 6 cm arc under 3 N load. Also run ID on the VITE trajectory to show that ID is only as good as the plan it receives.**Your tasks:** Fill in the TODO lines to:1. Apply IK at each timestep2. Compute the inertial torque M·q̈3. Compute the load compensation torque J^T·F_load4. Compute the net muscle torque

In [ ]:
# Step 1: Desired trajectory
dt_id = 0.001
t_id, pos_des = minjerk_arc(R_ARC, dt=dt_id)
n_id = len(t_id)

# Step 2: Inverse kinematics
q_des = np.zeros((n_id, 2))
for i in range(n_id):
    # TODO 1: Apply IK to the desired position
    # ← YOUR CODE HERE

# Step 3: Differentiate
qd_des = np.zeros_like(q_des); qdd_des = np.zeros_like(q_des)
qd_des[1:-1] = (q_des[2:] - q_des[:-2]) / (2*dt_id)
qd_des[0] = qd_des[1]; qd_des[-1] = qd_des[-2]
qdd_des[1:-1] = (q_des[2:] - 2*q_des[1:-1] + q_des[:-2]) / dt_id**2
qdd_des[0] = qdd_des[1]; qdd_des[-1] = qdd_des[-2]

# Steps 4-6: Inverse dynamics
tau_muscles = np.zeros((n_id, 2)); tau_inertial = np.zeros((n_id, 2))
tau_coriolis = np.zeros((n_id, 2)); tau_load = np.zeros((n_id, 2))

for i in range(n_id):
    M = mass_matrix(q_des[i]); C = coriolis(q_des[i], qd_des[i])
    J = arm.jacobian(q_des[i])
    # TODO 2: Compute inertial torque
    # ← YOUR CODE HERE
    tau_coriolis[i] = C
    # TODO 3: Compute load compensation torque
    # ← YOUR CODE HERE
    # TODO 4: Net muscle torque = dynamics - load
    # ← YOUR CODE HERE

# Step 7: Forward simulation
state = np.concatenate([Q_REF.copy(), [0.0, 0.0]])
hand_id_mj = np.zeros((n_id, 2))
for i in range(n_id):
    hand_id_mj[i] = fk(state[:2])
    tau_sim = tau_muscles[i] + tau_load[i]
    def deriv(s, _tau=tau_sim):
        qdd = joint_accelerations(s[:2], s[2:], _tau)
        return np.array([s[2], s[3], qdd[0], qdd[1]])
    state = rk4_step(state, deriv, dt_id)

# Also run ID on VITE trajectory for comparison
t_vt_d, pos_vt_d = vite_arc(R_ARC, dt=0.001)
t_vt_1ms = np.arange(0, T_MOVE+0.001, 0.001)
pos_vt_1ms = np.zeros((len(t_vt_1ms), 2))
for j in range(2): pos_vt_1ms[:, j] = np.interp(t_vt_1ms, t_vt_d, pos_vt_d[:, j])

def run_id(pos_in, dt, F_ext=None):
    n = len(pos_in)
    q = np.zeros((n,2)); qd = np.zeros_like(q); qdd = np.zeros_like(q)
    for i in range(n): q[i] = ik(pos_in[i,0], pos_in[i,1])
    qd[1:-1] = (q[2:]-q[:-2])/(2*dt); qd[0]=qd[1]; qd[-1]=qd[-2]
    qdd[1:-1] = (q[2:]-2*q[1:-1]+q[:-2])/dt**2; qdd[0]=qdd[1]; qdd[-1]=qdd[-2]
    tau = np.zeros((n,2))
    for i in range(n):
        M = mass_matrix(q[i]); C = coriolis(q[i], qd[i])
        tau[i] = M@qdd[i]+C
        if F_ext is not None: tau[i] -= arm.jacobian(q[i]).T @ F_ext
    st = np.concatenate([Q_REF.copy(),[0.,0.]]); hand = np.zeros((n,2))
    for i in range(n):
        hand[i] = fk(st[:2])
        t_s = tau[i] + (arm.jacobian(st[:2]).T @ F_ext if F_ext is not None else 0)
        def d(s, _t=t_s):
            a = joint_accelerations(s[:2],s[2:],_t)
            return np.array([s[2],s[3],a[0],a[1]])
        st = rk4_step(st, d, dt)
    return hand

hand_id_vt = run_id(pos_vt_1ms, 0.001, F_LOAD)

pd_id_mj = max_path_deviation(hand_id_mj, des6)*100
pd_id_vt = max_path_deviation(hand_id_vt, des6)*100
r6 = results[R_ARC]
print(f'ID + min-jerk 3N: {pd_id_mj:.4f} cm')
print(f'ID + VITE 3N:     {pd_id_vt:.2f} cm')
print(f'EPH opt 3N:       {r6["pd_opt"]:.2f} cm')

In [ ]:
# ═══════════════════════════════════════════════
# FIGURE 3 (Lecture §3): ID pipeline, 6 panels
# ═══════════════════════════════════════════════
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

axes[0,0].plot(pos_des[:,0]*100, pos_des[:,1]*100, TEAL, lw=2.5)
axes[0,0].set_xlabel('X (cm)'); axes[0,0].set_ylabel('Y (cm)')
axes[0,0].set_title('A. Step 1: P(t)'); axes[0,0].set_aspect('equal')

axes[0,1].plot(t_id*1000, np.degrees(q_des[:,0]), TEAL, lw=2, label='θ₁')
axes[0,1].plot(t_id*1000, np.degrees(q_des[:,1]), ORANGE, lw=2, label='θ₂')
axes[0,1].set_xlabel('ms'); axes[0,1].set_ylabel('deg'); axes[0,1].set_title('B. Step 2: IK'); axes[0,1].legend()

axes[0,2].plot(t_id*1000, qdd_des[:,0], TEAL, lw=2, label='q̈₁')
axes[0,2].plot(t_id*1000, qdd_des[:,1], ORANGE, lw=2, label='q̈₂')
axes[0,2].set_xlabel('ms'); axes[0,2].set_ylabel('rad/s²'); axes[0,2].set_title('C. Step 3: q̈(t)'); axes[0,2].legend()

for jj, ax, title in [(0, axes[1,0], 'D. Shoulder'), (1, axes[1,1], 'E. Elbow')]:
    ax.plot(t_id*1000, tau_inertial[:,jj], TEAL, lw=2, label='Mq̈')
    ax.plot(t_id*1000, tau_coriolis[:,jj], ORANGE, lw=2, label='C')
    ax.plot(t_id*1000, tau_load[:,jj], RED, lw=2, ls='--', label='J^T F')
    ax.plot(t_id*1000, tau_muscles[:,jj], NAVY, lw=2, ls=':', label='τ_muscles')
    ax.set_xlabel('ms'); ax.set_ylabel('N·m'); ax.set_title(title); ax.legend(fontsize=7)

# Panel F: Result — matching lecture Figure 3F
ax = axes[1,2]
ax.plot(des6[:,0]*100, des6[:,1]*100, 'gray', lw=2.5, alpha=0.35, label='Desired')
ax.plot(hand_id_mj[:,0]*100, hand_id_mj[:,1]*100, TEAL, lw=2.5, label=f'ID+MJ ({pd_id_mj:.3f} cm)')
ax.plot(hand_id_vt[:,0]*100, hand_id_vt[:,1]*100, PURPLE, lw=2, ls='--', label=f'ID+VITE ({pd_id_vt:.2f} cm)')
ax.plot(r6['h_opt'][:,0]*100, r6['h_opt'][:,1]*100, ORANGE, lw=2, ls='-.',
        label=f"EPH opt ({r6['pd_opt']:.2f} cm)")
ax.set_xlabel('X (cm)'); ax.set_ylabel('Y (cm)')
ax.set_title('F. Step 7: Result (3 N load)'); ax.legend(fontsize=7); ax.set_aspect('equal')

plt.tight_layout(); plt.show()

---## Part 4: Continuous Equilibrium Tracking (Lecture §4 — Figure 4)Give EPH the min-jerk arc as a continuous plan — update λ at every timestep via IK. Optimize both dy and C via grid search for each condition. Compare against discrete EPH and ID.**Your task:** Fill in the TODO line to compute λ(t) at every timestep.

In [ ]:
dt_cont = 0.0005
t_mj_fine, pos_mj_fine = minjerk_arc(R_ARC, dt=dt_cont)
n_mj_fine = len(t_mj_fine)

def eph_continuous(R, dy=0, C=0.25, perturb_fn=None, dt=0.0005):
    t_mj_c, pos_mj_c = minjerk_arc(R, dt=dt)
    pos_shifted = pos_mj_c.copy(); pos_shifted[:, 1] += dy
    n_c = len(t_mj_c)
    # Check reachability
    for k in range(0, n_c, max(1, n_c//20)):
        try:
            q = ik(pos_shifted[k,0], pos_shifted[k,1])
            if not arm.in_joint_limits(q): return None
        except: return None
    def lam_fn(t):
        if t >= T_MOVE: q_d = ik(pos_shifted[-1,0], pos_shifted[-1,1])
        else:
            idx = min(int(t/dt), n_c-1)
            q_d = ik(pos_shifted[idx,0], pos_shifted[idx,1])
        # TODO 1: Return lambda_for_posture with the desired joint angles and C
        # ← YOUR CODE HERE
    _, _, h, _ = simulate_lambda(lam_fn, T=1.2, dt=dt, perturb_fn=perturb_fn)
    return h

# Grid search for continuous EPH (dy + C), optimizing path deviation
# Same caveat: this is a brute-force search, not what the brain does.
def grid_search_continuous(R, perturb_fn=perturb_3N):
    des = dense_arc(R)
    best = 999; bdy = 0; bC = 0.25
    for dy in np.linspace(0, 0.08, 12):
        for C in np.linspace(0.20, 0.60, 10):
            h = eph_continuous(R, dy=dy, C=C, perturb_fn=perturb_fn, dt=0.0005)
            if h is None: continue
            v = max_path_deviation(h, des)
            if v < best: best = v; bdy = dy; bC = C
    return bdy, bC, best

# Discrete optimized (from Part 2)
disc_results = {int(R*100): (results[R]['dy'], results[R]['C'], results[R]['pd_opt']) for R in RADII}

# Continuous optimized
cont_results = {}
for R in RADII:
    dy_c, C_c, pd_c = grid_search_continuous(R)
    cont_results[int(R*100)] = (dy_c, C_c, pd_c*100)
    print(f'R={R*100:.0f}: cont opt dy={dy_c*100:.1f}, C={C_c:.2f}, pd={pd_c*100:.2f} cm')

In [ ]:
# ═══════════════════════════════════════════════
# FIGURE 4 (Lecture §4): Discrete vs Continuous vs ID
# ═══════════════════════════════════════════════
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Panel A: 6 cm arc paths
R = R_ARC; Rcm = int(R*100)
des = dense_arc(R)
dy_d, C_d = disc_results[Rcm][0], disc_results[Rcm][1]
dy_c, C_c = cont_results[Rcm][0], cont_results[Rcm][1]
_, h_disc, _ = eph_arc(R, dy=dy_d, C=C_d, perturb_fn=perturb_3N)
h_cont = eph_continuous(R, dy=dy_c, C=C_c, perturb_fn=perturb_3N)
pd_d = max_path_deviation(h_disc, des)*100
pd_c = max_path_deviation(h_cont, des)*100

ax1.plot(des[:,0]*100, des[:,1]*100, 'gray', lw=2.5, alpha=0.35, label='Desired')
ax1.plot(h_disc[:,0]*100, h_disc[:,1]*100, ORANGE, lw=2, ls='-.', label=f'Discrete opt ({pd_d:.2f} cm)')
ax1.plot(h_cont[:,0]*100, h_cont[:,1]*100, TEAL, lw=2.5, label=f'Continuous opt ({pd_c:.2f} cm)')
ax1.plot(hand_id_mj[:,0]*100, hand_id_mj[:,1]*100, NAVY, lw=2, ls='--', label=f'ID+MJ ({pd_id_mj:.3f} cm)')
ax1.plot(start_hand[0]*100, start_hand[1]*100, 'ko', ms=8, zorder=5)
ax1.plot(end6[0]*100, end6[1]*100, 'k*', ms=11, zorder=5)
ax1.set_xlabel('X (cm)'); ax1.set_ylabel('Y (cm)')
ax1.set_title('A. R = 6 cm, 3 N load'); ax1.legend(fontsize=8); ax1.set_aspect('equal')

# Panel B: Bar chart across radii
x = np.arange(3); w = 0.25
pd_disc_all = [disc_results[int(R*100)][2] for R in RADII]
pd_cont_all = [cont_results[int(R*100)][2] for R in RADII]
pd_id_all = []
for R in RADII:
    h_i = run_id(minjerk_arc(R, dt=0.001)[1], 0.001, F_LOAD)
    pd_id_all.append(max_path_deviation(h_i, dense_arc(R))*100)

ax2.bar(x-w, pd_disc_all, w, color=ORANGE, alpha=0.8, label='Discrete opt')
ax2.bar(x, pd_cont_all, w, color=TEAL, alpha=0.8, label='Continuous opt')
ax2.bar(x+w, pd_id_all, w, color=NAVY, alpha=0.8, label='ID + min-jerk')
ax2.set_xticks(x); ax2.set_xticklabels([f'R={R*100:.0f}' for R in RADII])
ax2.set_ylabel('Max path deviation (cm)'); ax2.set_title('B. Comparison across radii (3 N)')
ax2.legend(fontsize=8)
for i in range(3):
    ax2.text(i-w, pd_disc_all[i]+0.04, f'{pd_disc_all[i]:.2f}', ha='center', fontsize=7, color=ORANGE)
    ax2.text(i, pd_cont_all[i]+0.04, f'{pd_cont_all[i]:.2f}', ha='center', fontsize=7, color=TEAL)
    ax2.text(i+w, pd_id_all[i]+0.04, f'{pd_id_all[i]:.3f}', ha='center', fontsize=7, color=NAVY)

plt.tight_layout(); plt.show()

---## Part 5: Static Optimization (Lecture §5–6 — Figure 6)Distribute τ among 6 muscles: minimize Σ(Fᵢ/ρᵢ)³ subject to R·F = τ, F ≥ 0.**Your tasks:** Fill in the TODO lines to:1. Return the Crowninshield cost function2. Run the optimizer on the loaded torques

In [ ]:
R_mat = np.array([[m[3] for m in MUSCLE_DEFS], [m[4] for m in MUSCLE_DEFS]])
rho = np.array([m[1] for m in MUSCLE_DEFS])
mnames = ['Pec','BicL','BicS','Delt','TriL','TriLg']
mcols = [RED, ORANGE, '#E8735A', TEAL, '#3498db', GREEN]
n_power = 3

def static_optimization(tau_req):
    def cost(F):
        # TODO 1: Return the Crowninshield cost (sum of (F/rho)^n_power)
        # ← YOUR CODE HERE
    def cost_jac(F):
        return n_power * (F / rho) ** (n_power - 1) / rho
    constraints = {'type': 'eq', 'fun': lambda F: R_mat @ F - tau_req, 'jac': lambda F: R_mat}
    res = scipy_minimize(cost, np.ones(6)*0.1, jac=cost_jac, method='SLSQP',
                         bounds=[(0,None)]*6, constraints=constraints, options={'maxiter':500,'ftol':1e-12})
    return res.x if res.success else np.zeros(6)

# No-load torques
tau_noload = np.zeros((n_id, 2))
for i in range(n_id):
    M = mass_matrix(q_des[i]); C = coriolis(q_des[i], qd_des[i])
    tau_noload[i] = M @ qdd_des[i] + C

F_opt_nl = np.zeros((n_id, 6)); F_opt_ld = np.zeros((n_id, 6))
for i in range(n_id):
    if np.linalg.norm(tau_noload[i]) > 1e-6:
        F_opt_nl[i] = static_optimization(tau_noload[i])
    if np.linalg.norm(tau_muscles[i]) > 1e-6:
        # TODO 2: Run static optimization on the loaded muscle torques
        # ← YOUR CODE HERE

# ── FIGURE 6 ──
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
i_pk_nl = np.argmax(np.linalg.norm(tau_noload, axis=1))
i_pk_ld = np.argmax(np.linalg.norm(tau_muscles, axis=1))
x = np.arange(6); w = 0.35

axes[0].bar(x-w/2, F_opt_nl[i_pk_nl], w, color=TEAL, label='No load')
axes[0].bar(x+w/2, F_opt_ld[i_pk_ld], w, color=RED, label='3 N load')
axes[0].set_xticks(x); axes[0].set_xticklabels(mnames)
axes[0].set_ylabel('Force (N)'); axes[0].set_title('A. Peak: no load vs 3 N'); axes[0].legend()
for j in range(6):
    diff = F_opt_ld[i_pk_ld,j] - F_opt_nl[i_pk_nl,j]
    if abs(diff) > 2:
        axes[0].text(j, max(F_opt_nl[i_pk_nl,j],F_opt_ld[i_pk_ld,j])+1,
                     f'+{diff:.0f}N' if diff>0 else f'{diff:.0f}N', ha='center', fontsize=7, color=RED)

for j in range(6): axes[1].plot(t_id*1000, F_opt_nl[:,j], color=mcols[j], lw=1.5, label=mnames[j])
axes[1].set_xlabel('ms'); axes[1].set_ylabel('Force (N)'); axes[1].set_title('B. Full traj, no load')
axes[1].legend(fontsize=7, ncol=2)

for j in range(6): axes[2].plot(t_id*1000, F_opt_ld[:,j], color=mcols[j], lw=1.5, label=mnames[j])
axes[2].set_xlabel('ms'); axes[2].set_ylabel('Force (N)'); axes[2].set_title('C. Full traj, 3 N load')
axes[2].legend(fontsize=7, ncol=2)

plt.tight_layout(); plt.show()

---## Part 6: ID vs EPH — Head to Head (Lecture §7 — Figure 7)Compare normalized muscle activations: ID (static optimization forces) vs EPH (threshold displacements). Four traces per muscle: ID no-load, ID loaded, EPH no-load, EPH loaded.**Your tasks:** Fill in the TODO lines to:1. Normalize ID forces to [0, 1]2. Normalize EPH activations to [0, 1]

In [ ]:
# Resample EPH activations to 1 ms
# Re-run EPH to get activations at fine dt (eph_arc returns t, hand, acts)
t_eph0_full, _, a_eph0_full = eph_arc(R_ARC, dy=0, C=0.25, dt=0.0002)
t_eph3_full, _, a_eph3_full = eph_arc(R_ARC, dy=0, C=0.25, perturb_fn=perturb_3N, dt=0.0002)

a_e0_1ms = np.zeros((n_id, 6)); a_e3_1ms = np.zeros((n_id, 6))
for j in range(6):
    a_e0_1ms[:, j] = np.interp(t_id, t_eph0_full, a_eph0_full[:, j])
    a_e3_1ms[:, j] = np.interp(t_id, t_eph3_full, a_eph3_full[:, j])

def normalize_01(arr):
    out = np.zeros_like(arr)
    for j in range(arr.shape[1]):
        mx = arr[:, j].max()
        if mx > 0: out[:, j] = arr[:, j] / mx
    return out

# TODO 1: Normalize ID forces to [0,1] per muscle
# ← YOUR CODE HERE

# TODO 2: Normalize EPH activations to [0,1] per muscle
# ← YOUR CODE HERE

# ── FIGURE 7: 2×3, 4 traces per muscle ──
mnames_full = ['Pectoralis','Biceps Long','Biceps Short','Deltoid','Triceps Long','Triceps Lat.']
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for j, ax in enumerate(axes.flat):
    ax.plot(t_id*1000, a_id_nl[:,j], TEAL, lw=2, label='ID, 0N')
    ax.plot(t_id*1000, a_id_ld[:,j], NAVY, lw=2, ls='--', label='ID, 3N')
    ax.plot(t_id*1000, a_eph_nl[:,j], GREEN, lw=1.5, alpha=0.7, label='EPH, 0N')
    ax.plot(t_id*1000, a_eph_ld[:,j], RED, lw=1.5, ls=':', alpha=0.7, label='EPH, 3N')
    ax.set_title(mnames_full[j], fontweight='bold', color=mcols[j])
    ax.set_xlim(0, 900); ax.set_ylim(-0.05, 1.1)
    if j >= 3: ax.set_xlabel('Time (ms)')
    if j % 3 == 0: ax.set_ylabel('Normalized activation')
    if j == 0: ax.legend(fontsize=6.5)
plt.suptitle('ID vs. EPH: Muscle activations on 6 cm arc', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

---## Summary1. **EPH hits a wall on curved paths** (Part 1): Even with grid-searched R+C, EPH deviates ≈1.6 cm from the arc. Min-jerk tracks it at ≈0.02 cm. The grid search finds the *theoretical best* — but the brain would need to learn these parameters from experience for every trajectory and load combination.2. **The gap is consistent across radii** (Part 2): Min-jerk is 67–137× better than optimized EPH. The optimal R-shift changes with both trajectory shape and load magnitude.3. **The ID pipeline bridges the gap** (Part 3): IK → differentiate → τ = Mq̈ + C − J^T·F → forward sim. Sub-millimeter tracking. But ID is only as good as its trajectory: ID+VITE (≈1.8 cm) is comparable to optimized EPH.4. **A continuous planner helps but can't close the gap** (Part 4): Continuous λ tracking halves the error but remains 7–14× worse than ID. The dynamics lag — springs reacting to error vs ID anticipating it — is the irreducible bottleneck.5. **Static optimization solves the redundancy problem** (Part 5): The Crowninshield criterion distributes torques among muscles. The load changes *which* muscles are recruited (TriL: 0 → 29 N).6. **ID and EPH predict fundamentally different muscle patterns** (Part 6): ID is phasic, selective, with minimal co-contraction. EPH is sustained, broad, with substantial co-contraction. Under load, ID recruits specific muscles; EPH scales globally. These are testable EMG predictions for the Week 11 Duel.